In [1]:
import os
import time
import joblib
import pandas as pd

from openpyxl import Workbook
from sklearn.ensemble import GradientBoostingRegressor

# 所有数据集名称
feature_name_ls = [
    'FULL_elements1',
    'FULL_magpie2',
    'FULL_merdig3',
    'FULL_magpie&Morphology4',
    'FULL_magpie&WenAlloys5',
    'FULL_magpie&Merdig6',
    'FULL_Magpie&Wen&Merdig&Mor7',
    'FULL_8',
    'FULL_magpie&merdig&mor&formula9'
]

results = []

for feature_name in feature_name_ls:
    print(f"Processing {feature_name}...")
    
    # 路径准备
    data_path = f'data/{feature_name}.xlsx'
    model_path = f'{feature_name}/models/gboost/qf_model1.pkl'

    if not os.path.exists(data_path) or not os.path.exists(model_path):
        print(f"Skipped {feature_name} (file not found)")
        continue

    try:
        # 读取数据（全部样本，去除目标列）
        df = pd.read_excel(data_path, index_col=0).dropna()
        df_X = df.iloc[:, :-3]  # 假设后3列是目标或无关特征

        # 读取模型
        model = joblib.load(model_path)

        # 预测并计时（全体样本）
        start = time.perf_counter()
        _ = model.predict(df_X)
        end = time.perf_counter()
        total_latency = (end - start)
        avg_latency_ms = total_latency / len(df_X) * 1000  # 平均每个样本的耗时

        # 模型信息
        model_size_mb = os.path.getsize(model_path) / 1024 / 1024
        num_trees = len(model.estimators_)
        avg_tree_depth = sum(tree[0].tree_.max_depth for tree in model.estimators_) / num_trees

        # 记录结果
        results.append({
            "Dataset": feature_name,
            "Samples": len(df_X),
            "Total Latency (s)": round(total_latency, 4),
            "Avg Latency (ms/sample)": round(avg_latency_ms, 4),
            "Model Size (MB)": round(model_size_mb, 2),
            "Number of Trees": num_trees,
            "Average Tree Depth": round(avg_tree_depth, 2)
        })

    except Exception as e:
        print(f"❌ Error processing {feature_name}: {e}")

# 保存为 Excel 表格
df_result = pd.DataFrame(results)
df_result.to_excel("model_computational_cost.xlsx", index=False)

print("\n✅ 所有计算完成，结果已保存为 model_computational_cost.xlsx")


Processing FULL_elements1...
Processing FULL_magpie2...
Processing FULL_merdig3...
Processing FULL_magpie&Morphology4...
Processing FULL_magpie&WenAlloys5...
Processing FULL_magpie&Merdig6...
Processing FULL_Magpie&Wen&Merdig&Mor7...
Processing FULL_8...
Processing FULL_magpie&merdig&mor&formula9...

✅ 所有计算完成，结果已保存为 model_computational_cost.xlsx


In [ ]:
import os
import time
import joblib
import pandas as pd
import numpy as np
def evaluate_model(folder_path, weights, X):
    total_latency = 0.0
    total_model_size = 0.0
    total_trees = 0
    total_depth = 0.0
    total_models = 0
    sample_count = len(X)

    start = time.perf_counter()
    for model_name, weight in weights.items():
        for i in range(5):
            model_file = os.path.join(folder_path, f'{model_name}_{i}.pkl')
            if not os.path.exists(model_file):
                print(f"Model {model_file} not found.")
                continue
            model = joblib.load(model_file)
            _ = model.predict(X)  # 预测

            # 模型统计
            total_models += 1
            total_model_size += os.path.getsize(model_file) / 1024 / 1024  # MB

            if hasattr(model, 'estimators_'):
                estimators = model.estimators_
                # 判断是否为二维数组（GradientBoostingRegressor）
                if isinstance(estimators[0], (list, np.ndarray)):
                    n_trees = len(estimators)
                    total_trees += n_trees
                    for tree in estimators:
                        total_depth += tree[0].tree_.max_depth
                else:  # 一维数组（RandomForestRegressor）
                    n_trees = len(estimators)
                    total_trees += n_trees
                    for tree in estimators:
                        total_depth += tree.tree_.max_depth
            else:
                print(f"Warning: Model {model_file} has no estimators_ attribute.")

    end = time.perf_counter()

    avg_latency_ms = (end - start) / sample_count * 1000
    avg_model_size = total_model_size / total_models if total_models else 0
    avg_tree_count = total_trees / total_models if total_models else 0
    avg_tree_depth = total_depth / total_trees if total_trees else 0

    return {
        "Samples": sample_count,
        "Total Latency (s)": round(end - start, 4),
        "Avg Latency (ms/sample)": round(avg_latency_ms, 4),
        "Model Size (MB)": round(avg_model_size, 2),
        "Number of Trees": int(avg_tree_count),
        "Average Tree Depth": round(avg_tree_depth, 2)
    }



# ---------- Small Model ----------
df_small = pd.read_excel('G:\AIMS_Lab_final\my_project\AAA-MgAlloy\AI_Alloy\Small_Data_new/20240607FULL_final.xlsx', index_col=0, sheet_name='qf_21')
X_small = df_small.iloc[:, :-2]

folder_path_small = "G:\AIMS_Lab_final\my_project\AAA-MgAlloy\AI_Alloy\model_analysis\small_model/final_model/qf/rf"
weights_small = {'rf': 0.2}
results_small = evaluate_model(folder_path_small, weights_small, X_small)
results_small["Model"] = "Small Model"

# ---------- General Model ----------
df_general = pd.read_excel('G:\AIMS_Lab_final\my_project\AAA-MgAlloy\AI_Alloy\model_analysis\general_model/qf_models/train_set_new.xlsx', index_col=0)
X_general = df_general.drop(columns=['Precipitate Distribution', 'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])

folder_path_general = "G:\AIMS_Lab_final\my_project\AAA-MgAlloy\AI_Alloy\model_analysis\general_model/qf_models"
weights_general = {'xgboost': 0.7 / 5, 'rf': 0.3 / 5}
results_general = evaluate_model(folder_path_general, weights_general, X_general)
results_general["Model"] = "General Model"

# 合并结果并保存
df_result = pd.DataFrame([results_small, results_general])
df_result = df_result[["Model", "Samples", "Total Latency (s)", "Avg Latency (ms/sample)", "Model Size (MB)", "Number of Trees", "Average Tree Depth"]]
df_result.to_excel("computational_cost_qf_small_vs_general.xlsx", index=False)

print("✅ Small model & General model 推理性能评估完成！结果保存为 computational_cost_qf_small_vs_general.xlsx")


KeyError: "['Yield_Strength', 'Tensile_Strength (UTS)'] not found in axis"